In [ ]:
import pandas as pd
import numpy as np

# 🔹 Load Excel file
file_path = "PLAN_ACTUAL.xlsx"   # change to your file name
df = pd.read_excel(file_path)

# 🔹 Separate material column
materials = df['Material']

# 🔹 Take only demand columns (all except Material)
demand_data = df.drop(columns=['Material'])

# 🔹 Calculate statistics row-wise
mean_demand = demand_data.mean(axis=1)
std_dev = demand_data.std(axis=1, ddof=1)
count = demand_data.count(axis=1)

# 🔹 Standard Error
std_error = std_dev / np.sqrt(count)

# 🔹 95% Confidence Interval
z = 1.96
lower_ci = mean_demand - z * std_error
upper_ci = mean_demand + z * std_error

# 🔹 Create result dataframe
result = pd.DataFrame({
    'Material': materials,
    'Mean_Demand': mean_demand,
    'Std_Dev': std_dev,
    'Lower_95_CI': lower_ci,
    'Upper_95_CI': upper_ci
})

print(result)

# 🔹 Optional: Save to Excel
result.to_excel("Confidence_Interval_Output.xlsx", index=False)


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# 🔹 Load Excel file
file_path = "PLAN_ACTUAL.xlsx"   # change to your file name
df = pd.read_excel(file_path)

# 🔹 Separate material column
materials = df['Material']

# 🔹 Take only demand columns (all except Material)
demand_data = df.drop(columns=['Material'])

# 🔹 Calculate statistics row-wise
mean_demand = demand_data.mean(axis=1)
std_dev = demand_data.std(axis=1, ddof=1)
count = demand_data.count(axis=1)

# 🔹 Standard Error
std_error = std_dev / np.sqrt(count)

# 🔹 95% Confidence Interval using t-score
alpha = 0.05
# t critical value depends on df = count - 1
t_values = stats.t.ppf(1 - alpha/2, df=count - 1)

lower_ci = mean_demand - t_values * std_error
upper_ci = mean_demand + t_values * std_error

# 🔹 Create result dataframe
result = pd.DataFrame({
    'Material': materials,
    'Mean_Demand': mean_demand,
    'Std_Dev': std_dev,
    'Lower_95_CI': lower_ci,
    'Upper_95_CI': upper_ci
})

print(result)

# 🔹 Optional: Save to Excel
result.to_excel("Confidence_Interval_Output.xlsx", index=False)


In [ ]:
import pandas as pd
import re

# ===============================
# LOAD FILE
# ===============================

df = pd.read_excel("combined_file.xlsx")

# ===============================
# EXTRACT ALL DATES
# ===============================

columns = df.columns.tolist()

dates = sorted(
    list(set(re.findall(r"\d{4}-\d{2}-\d{2}", " ".join(columns))))
)

# ===============================
# DAILY DEVIATION
# ===============================

for date in dates:
    
    indent_col = f"{date} Indent"
    actual_col = f"{date} Total Production Plan"
    
    if indent_col in df.columns and actual_col in df.columns:
        
        # Daily deviation
        df[f"{date} Deviation"] = df[actual_col] - df[indent_col]
        
        # Percentage deviation
        df[f"{date} Deviation_%"] = (
            df[f"{date} Deviation"] / df[indent_col]
        ) * 100

# ===============================
# MONTH TILL DATE SUMMARY
# ===============================

indent_cols = [
    f"{d} Indent"
    for d in dates
    if f"{d} Indent" in df.columns
]

actual_cols = [
    f"{d} Total Production Plan"
    for d in dates
    if f"{d} Total Production Plan" in df.columns
]

df["Total_Indent_Till_Date"] = df[indent_cols].sum(axis=1)
df["Total_Actual_Till_Date"] = df[actual_cols].sum(axis=1)

df["Total_Deviation"] = (
    df["Total_Actual_Till_Date"] - df["Total_Indent_Till_Date"]
)

df["Total_Deviation_%"] = (
    df["Total_Deviation"] / df["Total_Indent_Till_Date"]
) * 100

# ===============================
# SAVE
# ===============================

df.to_excel("indent_vs_actual_wide_analysis.xlsx", index=False)

print("Indent vs Actual comparison completed successfully.")
